# Pakistan Consumer Voice Intelligence | Industry-Style Project 1
**Real Roman Urdu sentiment corpus · 2,000 original-source records · Pakistan fashion-adjacent decision support**

**Business case:** A Pakistani clothing brand wants to understand how Roman Urdu consumer sentiment is expressed, what apparel-related concerns recur, and how to establish a reliable listening workflow before investing in platform campaigns.

**Truth-in-data:** This is **not a verified Instagram export**, and the 161 apparel-keyword matches are **unreviewed candidates**, not 161 confirmed clothing reviews. Original source data was collected from a mix of reviews/comments; timestamps, brands, platform, Pakistan location, follower counts, likes, shares, paid ads, clicks, and reach are **not recorded**. No such values are fabricated. This is an **offline consumer-language pilot**, not a measurement of the brand's current Instagram audience.

**Dataset:** Sharf, Z. (2017), *Roman Urdu Data Set*, UCI Machine Learning Repository, DOI: [10.24432/C58325](https://doi.org/10.24432/C58325). Original source: https://archive.ics.uci.edu/dataset/458/roman+urdu+data+set . Repository's raw corpus is mirrored at https://github.com/Smat26/Roman-Urdu-Dataset . See `DATA_PROVENANCE.md` in the project bundle.

**How to run:** Upload both this `.ipynb` and `pakistan_roman_urdu_real_2000.csv` to Colab (or upload CSV when prompted). Execute Runtime → Run all. **The CSV is the actual data input, not generated demo rows.**

## 1 · Business questions and success criteria
1. What do original sentiment labels indicate across the 2,000-record public Roman Urdu pilot?
2. What is the observed sentiment mix of the **candidate** apparel subset, with sample sizes and uncertainty made explicit?
3. What recurring apparel-experience themes could a Pakistani clothing brand **investigate** further?
4. Can a held-out text classifier outperform a majority baseline without duplicate/leakage contamination?
5. When does a larger workload make Pandas slower, and how would we decide whether Spark is justified?

**Acceptance criteria:** data dictionary/provenance; count, types, missing/duplicate audit; summary statistics; ≥2 useful charts; two observed findings and an actionable, qualified hypothesis; 2k→20k→200k benchmark; conceptual classification/clustering/regression mapping. Include limitations and the 10-mark course rubric at the end.

In [ ]:
# Python 3.10+ / Colab: install only mainstream pinned-minimum packages.
%pip -q install "pandas>=2.0" "numpy>=1.24" "matplotlib>=3.7" "seaborn>=0.12" "scikit-learn>=1.3" "joblib>=1.3" "psutil>=5.9" "nbformat>=5.8"

In [ ]:
import os, re, json, time, platform, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
import joblib, psutil
RANDOM_STATE=42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option('display.max_colwidth', 90)
OUTPUT=Path('pakistan_fashion_intelligence_outputs'); OUTPUT.mkdir(exist_ok=True)
print('Python:',platform.python_version(),'Pandas:',pd.__version__, 'Memory GB:',round(psutil.virtual_memory().total/1e9,1))

## 2 · Load **real**, separate CSV
A CSV is mandatory, rather than invented posts. Set `CSV_PATH` to a file or upload it interactively. A local source fingerprint is printed for audit. If the CSV is missing, execution stops instead of silently generating synthetic records.

In [ ]:
CSV_PATH = 'pakistan_roman_urdu_real_2000.csv'  # Also works as /content/drive/MyDrive/.../filename.csv
if not Path(CSV_PATH).exists():
    try:
        from google.colab import files
        print('Upload pakistan_roman_urdu_real_2000.csv')
        uploaded=files.upload()
        if 'pakistan_roman_urdu_real_2000.csv' not in uploaded:
            raise FileNotFoundError('Upload the supplied CSV, not the old synthetic dataset.')
    except ImportError:
        raise FileNotFoundError(f'CSV not found: {CSV_PATH}; place the supplied file beside the notebook')
raw_bytes=Path(CSV_PATH).read_bytes()
print('SHA256:',hashlib.sha256(raw_bytes).hexdigest(), 'Bytes:',len(raw_bytes))
raw=pd.read_csv(CSV_PATH,encoding='utf-8',dtype={'source_row_number':'Int64'})
required=['record_id','comment_text','sentiment','segment','source_row_number','source_dataset']
missing=set(required)-set(raw.columns)
assert not missing, f'Missing required fields: {missing}'
print('RAW SHAPE:',raw.shape)
display(raw.head(5))

### Data dictionary and observation unit
**One row:** an original Roman Urdu text item, sentiment label, source-row identifier, and derived keyword flag.

| Column | Meaning | Provenance |
|---|---|---|
| `record_id` | Deterministic original-row identifier, not a social platform ID | Derived |
| `comment_text` | Original recorded Roman Urdu comment/review (minimal privacy masking) | Source |
| `sentiment` | Positive / Neutral / Negative | Source label |
| `segment` | Apparel keyword candidate vs general Roman Urdu | Derived keyword screening, **not validated classification** |
| `source_row_number` | Raw source CSV row number for traceability | Derived lineage |
| `source_dataset` | UCI attribution | Added |

**Unavailable by design:** platform, follower count, location, demographic information, dates, engagement, impressions, ad spend, clicks, sales. This notebook refuses to infer them.

In [ ]:
audit=pd.DataFrame({'dtype':raw.dtypes.astype(str),'missing':raw.isna().sum(),'distinct':raw.nunique(dropna=False)})
display(audit)
print('Exact duplicates:',raw.duplicated().sum(),'Duplicate IDs:',raw.record_id.duplicated().sum())
print('Sentiments:',raw.sentiment.value_counts(dropna=False).to_dict())
print('Segments:',raw.segment.value_counts(dropna=False).to_dict())
assert len(raw)==2000, 'Expected the supplied 2,000-row real-data extract.'
assert raw.record_id.is_unique, 'Record lineage collision.'
assert raw.source_row_number.is_unique, 'One original source row should appear at most once.'
assert set(raw.sentiment.dropna()) <= {'Positive','Neutral','Negative'}
assert set(raw.segment.dropna()) <= {'apparel_keyword_candidate','general_roman_urdu'}

## 3 · Clean and validate, with an audit trail
Do not assume missing fields are zero; use explicit flags and fail on missing labels or source IDs. Comments have already been de-duplicated on normalized text during extract construction; we check again below. Preserve the original comment strings in `comment_text`.

In [ ]:
df=raw.copy()
for col in ['comment_text','record_id','segment','sentiment']:
    df[col]=df[col].astype('string').str.strip()
df['text_model']=df.comment_text.fillna('').str.replace(r'\s+',' ',regex=True)
valid=(df.text_model.str.len().ge(12)&df.sentiment.isin(['Positive','Neutral','Negative'])&df.record_id.notna()&df.source_row_number.notna())
invalid=df.loc[~valid].copy()
if len(invalid):
    display(invalid[['record_id','sentiment','comment_text']].head()); print('Excluded invalid:',len(invalid))
df=df.loc[valid].drop_duplicates(subset=['record_id']).copy()
df['text_key']=df.text_model.str.lower().str.replace(r'[^\w]+',' ',regex=True).str.strip()
assert not df.text_key.duplicated().any(), 'Duplicate normalized text found; inspect before modeling.'
df['char_count']=df.text_model.str.len().astype(int)
df['token_count']=df.text_model.str.split().str.len().astype(int)
df['apparel_candidate']=df.segment.eq('apparel_keyword_candidate')
df.to_csv(OUTPUT/'validated_source_records.csv',index=False,encoding='utf-8-sig')
print('Validated records:',len(df),'Apparel candidates:',int(df.apparel_candidate.sum()))
display(df[['char_count','token_count']].describe().T)
display(df.groupby('segment',observed=True)[['char_count','token_count']].agg(['count','median','mean']).round(1))

## 4 · Exploratory analysis (EDA): denominators first
The **total corpus** is a broad Roman Urdu sentiment language sample. Only the `apparel_keyword_candidate` slice is plausibly clothing-related. Its proportion is not market share. Differences in sentiment across these two cohorts may reflect the keyword filter and source mix rather than clothing consumers.

In [ ]:
overall=df.sentiment.value_counts().reindex(['Positive','Neutral','Negative'],fill_value=0)
by_segment=pd.crosstab(df.segment,df.sentiment).reindex(columns=['Positive','Neutral','Negative'],fill_value=0)
print('Overall sentiment counts:'); display(overall.rename('records').to_frame())
print('Cohort sentiment counts:'); display(by_segment)
print('Cohort sentiment % (row-normalized):'); display(by_segment.div(by_segment.sum(axis=1),axis=0).mul(100).round(2))
print('Typical text tokens (median):',df.token_count.median())
app=df[df.apparel_candidate].copy()
print('Apparel candidate sample:',len(app),'NOT manually verified.')

In [ ]:
palette={'Positive':'#237a57','Neutral':'#7c8796','Negative':'#c6574b'}
fig,axs=plt.subplots(1,2,figsize=(13,4.5))
overall.plot.bar(ax=axs[0],color=[palette[x] for x in overall.index]);axs[0].set(title='Label counts · full Roman Urdu corpus (n=2,000)',ylabel='Records',xlabel='')
by_segment.plot.bar(stacked=True,ax=axs[1],color=[palette[x] for x in by_segment.columns]);axs[1].set(title='Label distribution by screening cohort',ylabel='Records',xlabel='Screening cohort');axs[1].tick_params(axis='x',rotation=12)
plt.tight_layout();plt.savefig(OUTPUT/'01_sentiment_mix.png',dpi=160);plt.show()

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(13,4.5))
sns.histplot(df.token_count,bins=50,ax=axs[0],color="#527c9e");axs[0].set(title='Text length distribution',xlabel='Whitespace-separated tokens')
shares=by_segment.div(by_segment.sum(axis=1),axis=0).mul(100)
shares.plot.bar(ax=axs[1],color=[palette[x] for x in shares.columns]);axs[1].set(title='Within-cohort sentiment proportions',ylabel='Percent',xlabel='Cohort');axs[1].tick_params(axis='x',rotation=12)
plt.tight_layout();plt.savefig(OUTPUT/'02_text_length_and_sentiment.png',dpi=160);plt.show()

### Uncertainty and decision safeguards
Compute Wilson 95% confidence intervals for the observed fraction of negative-labeled items **conditional on this particular candidate subset**. This is an interval for a Bernoulli proportion under an illustrative sampling assumption. Because the sample is nonrandom and keyword-screened, it does **not** give a population-representative estimate for Pakistani fashion consumers.

In [ ]:
from math import sqrt

def wilson(k,n,z=1.96):
    if n<=0: return (np.nan,np.nan)
    p=k/n; den=1+z*z/n;center=(p+z*z/(2*n))/den;rad=z*sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return center-rad,center+rad
for cohort,part in [('All Roman Urdu',df),('Apparel keyword candidates',app)]:
    k=(part.sentiment=='Negative').sum();n=len(part);lo,hi=wilson(k,n)
    print(f'{cohort}: negative {k}/{n} ({k/n:.1%}); illustrative Wilson 95% [{lo:.1%},{hi:.1%}]')

## 5 · Candidate fashion topics (transparent keyword rules)
The phrases are **hypotheses for human review**; not a validated topic model, not counts of buyers, and not causal evidence. Roman Urdu spellings are inconsistent. Topic hits may overlap and can include false positives. Never automatically claim "best brand," "best platform," or "best posting time" from these counts.

In [ ]:
TOPIC_RULES={
  'Quality / material':r'\b(?:quality|fabric|material|kapr\w*|stuff|naram|silai|stitch\w*)\b',
  'Size / fit':r'\b(?:size|fit|fitting|tight|loose|bara|chota|choti|comfortable)\b',
  'Price / value':r'\b(?:price|qeemat|qimat|mehnga|mehanga|paisa|pese|rupee|discount|sale)\b',
  'Color / design':r'\b(?:color|colour|rang|design|print|style|stylish)\b',
  'Delivery / ordering':r'\b(?:deliver\w*|order|mangway\w*|receive|received|return|parcel)\b'
}
for topic,pattern in TOPIC_RULES.items():
    app[topic]=app.text_model.str.contains(pattern,case=False,regex=True,na=False)
topics=pd.DataFrame({'topic':list(TOPIC_RULES),'mentions':[int(app[k].sum()) for k in TOPIC_RULES]})
topics['cohort_n']=len(app)
topics['candidate_share_pct']=(100*topics.mentions/len(app)).round(1)
display(topics.sort_values('mentions',ascending=False))
ax=topics.sort_values('mentions').plot.barh(x='topic',y='mentions',legend=False,figsize=(9,4),color='#527c9e')
ax.set(title=f'Apparel-screened language cues · n={len(app)}',xlabel='Candidate comments matched (overlap allowed)',ylabel='')
plt.tight_layout();plt.savefig(OUTPUT/'03_apparel_themes.png',dpi=160);plt.show()
topics.to_csv(OUTPUT/'apparel_candidate_topic_flags_summary.csv',index=False)

### Review real excerpts (human validation recommended)
Keyword matching can confuse an apparel customer review with a joke, news quote, or sports comment. **Do not reproduce personally identifying content in public reports.** Manually inspect examples and record false positives before brand strategy. Human review is particularly important for sarcasm, negation, code-switching and topical relevance.

In [ ]:
sample_review=app[['record_id','comment_text','sentiment']].sample(min(12,len(app)),random_state=RANDOM_STATE)
display(sample_review)
sample_review.to_csv(OUTPUT/'manual_review_candidates.csv',index=False,encoding='utf-8-sig')
# Review protocol: assess apparel_relevant (yes/no/uncertain), sentiment_correct (yes/no/uncertain),
# and actionability (quality, fit, price, delivery, other); do not treat regex candidates as ground truth.

## 6 · Text classification | independent hold-out + baseline
We **train on original sentiment labels**, not engagement-derived labels. **No** engagement metric is reused as a prediction feature. Hold out 20% of texts, stratifying labels. Keep exact-text de-duplication before splitting; in production, add near-duplicate/author/time grouping and out-of-domain validation. Report macro-F1, weighted F1, per-class errors, and a dummy majority baseline. The evaluation reflects this mixed-source corpus, not Pakistani brand Instagram generalization.

In [ ]:
X=df.text_model.astype(str); y=df.sentiment.astype(str)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=RANDOM_STATE,stratify=y)
classifier=Pipeline([('tfidf',TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=2,max_df=.98,max_features=60000,sublinear_tf=True)),('model',LogisticRegression(max_iter=1200,class_weight='balanced',random_state=RANDOM_STATE))])
baseline=DummyClassifier(strategy='most_frequent');baseline.fit(X_train,y_train)
classifier.fit(X_train,y_train)
pred=classifier.predict(X_test);base_pred=baseline.predict(X_test)
print('Train / test:',len(X_train),len(X_test))
print('Majority baseline macro-F1:',round(f1_score(y_test,base_pred,average='macro'),3))
print('TF-IDF model macro-F1:',round(f1_score(y_test,pred,average='macro'),3),'Accuracy:',round(accuracy_score(y_test,pred),3))
print(classification_report(y_test,pred,digits=3,zero_division=0))
classes=['Positive','Neutral','Negative'];cm=confusion_matrix(y_test,pred,labels=classes)
fig,ax=plt.subplots(figsize=(6,5));sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=classes,yticklabels=classes,ax=ax)
ax.set(xlabel='Predicted',ylabel='True source label',title='Held-out confusion matrix');plt.tight_layout();plt.savefig(OUTPUT/'04_confusion_matrix.png',dpi=160);plt.show()
joblib.dump(classifier,OUTPUT/'roman_urdu_tfidf_sentiment.joblib')
pd.DataFrame({'text':X_test.values,'true_label':y_test.values,'prediction':pred}).to_csv(OUTPUT/'heldout_predictions.csv',index=False,encoding='utf-8-sig')

### Practical score / inference
`score_texts()` handles new Roman Urdu text. Test results can be inaccurate for Urdu script, out-of-domain fashion content, sarcastic comments, unfamiliar spellings, and evolving slang. A model confidence score is not a calibrated probability of customer satisfaction.

In [ ]:
def score_texts(texts):
    texts=pd.Series(texts,dtype='string').fillna('').astype(str)
    labels=classifier.predict(texts)
    probs=classifier.predict_proba(texts)
    return pd.DataFrame({'text':texts,'predicted_sentiment':labels,'max_model_probability':np.max(probs,axis=1).round(3)})
display(score_texts(['Kapra acha hai aur fitting perfect hai','Order late aya aur quality kharab thi','Nayi collection dekh raha hoon']))

## 7 · Reproducible workload scaling: 2k → 20k → 200k
**This is a synthetic workload repetition experiment using genuine seed rows**, never 200,000 independent Pakistani comments. No claims about market composition or unique posts follow from repeated data. Benchmark the same group-by operation, and measure memory of each expanded DataFrame. These sizes are configurable. Spark is **not required** for 2,000 rows.

In [ ]:
sizes=[2_000,20_000,200_000]
bench=[]
base=df[['sentiment','segment','token_count']].reset_index(drop=True)
for n in sizes:
    repeated=base.iloc[np.arange(n)%len(base)].copy()
    t0=time.perf_counter()
    summary=repeated.groupby(['segment','sentiment'],observed=True).token_count.agg(['count','median','mean'])
    elapsed=time.perf_counter()-t0
    mem=repeated.memory_usage(index=True,deep=True).sum()/1024**2
    bench.append({'rows_repeated_for_benchmark':n,'groupby_seconds':elapsed,'expanded_frame_MiB':mem,'distinct_original_records':len(base)})
    del repeated
benchmark=pd.DataFrame(bench);display(benchmark)
benchmark.to_csv(OUTPUT/'scaling_benchmark.csv',index=False)
fig,ax=plt.subplots(figsize=(7,4));ax.plot(benchmark.rows_repeated_for_benchmark,benchmark.groupby_seconds,marker='o')
ax.set(title='Pandas group-by timing · repeated-row workload',xlabel='Processed rows (not unique source records)',ylabel='Elapsed seconds');plt.tight_layout();plt.savefig(OUTPUT/'05_scaling_benchmark.png',dpi=160);plt.show()
print('Timing varies with hardware, runtime warmth, I/O, process memory, and dataset types. Not a universal Spark crossover.')

## 8 · Big Data bridge: OPTIONAL PySpark comparison
Run the next cell **only if PySpark is installed and you choose to compare it**; it does not silently install Java or start Spark. It measures a Spark group-by on 200k repeated rows and is not evidence of genuine 200k posts. A local Spark session can be slower because of startup/overhead; use distribution only when volume/compute/operational reliability justify it.

In [ ]:
RUN_SPARK=False  # Optional extension: set True after installing pyspark in Colab
if RUN_SPARK:
    try:
        from pyspark.sql import SparkSession,functions as F
        spark=SparkSession.builder.master('local[*]').appName('PakistanRomanUrduPilot').getOrCreate()
        spark_input=base.iloc[np.arange(200_000)%len(base)].copy()
        t0=time.perf_counter()
        spark_result=(spark.createDataFrame(spark_input).groupBy('segment','sentiment').agg(F.count('*').alias('n'),F.avg('token_count').alias('mean_tokens')).collect())
        print('Spark duration including frame conversion:',round(time.perf_counter()-t0,4),'s')
        print(spark_result)
        spark.stop()
    except ImportError:
        print('Install: %pip install pyspark ; restart/run again')
else:
    print('Optional Spark disabled: appropriate for 2k-row introductory assignment.')

## 9 · Analytics technique mapping and limitations
**Classification:** already demonstrated with original 3-class sentiment labels. **Clustering:** optional future TF-IDF / embeddings, manually validate discovered themes. **Regression:** would require a real numeric target like post-level reach, sales, or likes with source timestamps; **these targets are absent**, so predicting them here would be fiction. Do not train a high/low engagement classifier on engagement values used as input features.

**Industry pipeline checklist:** raw/source archive; deterministic extract and lineage; data validation; immutable holdout; model + baseline metrics; reusable inference; human-review queue; ethical/privacy checks; observable benchmarks; outputs and recommendations; monitoring and drift/label-quality plan.

**Limitations:** corpus collected in 2017, sources are mixed, sampling is not representative, labels have potential noise, Roman Urdu is not Urdu script, apparel subset is only keyword-screened, and no region/platform/campaign outcomes can be measured. Additional authorized first-party Instagram Insights or Meta exports would be needed to answer the original clothing brand engagement question.

## 10 · Evidence → tentative business actions
Generate observations using computed statistics, not fixed invented percentages. Turn these into hypotheses for the Pakistani clothing brand. Do not treat text sentiment as reach, engagement, market share, or campaign ROI.

In [ ]:
a=app.sentiment.value_counts();n=len(app);neg=int(a.get('Negative',0));pos=int(a.get('Positive',0))
leading=topics.sort_values('mentions',ascending=False).iloc[0]
review_text=f"""# Project 1 — Evidence-based executive readout

**Data source:** UCI Roman Urdu Data Set, 2,000 source-derived unique Roman Urdu comment/review records, NOT a verified Pakistani fashion Instagram feed.

**Observation 1:** Source labels across the complete pilot: {overall.to_dict()}. The cohort is intentionally mixed-source, so these percentages are sample descriptions only.

**Observation 2:** Among {n} *keyword-screened apparel candidates*, {neg} carried the source Negative label ({neg/n:.1%}) and {pos} the Positive label ({pos/n:.1%}). These are not manually confirmed brand reviews.

**Observation 3:** The most frequent apparel-topic screen is **{leading['topic']}** ({int(leading['mentions'])}/{n} candidate texts). Regex rules overlap and can misclassify.

**Proposed business test:** Have a native Roman Urdu reader verify an audited random sample of the apparel candidates, identify which quality/fit/price/delivery concerns genuinely recur, and run a controlled Instagram or social-listening experiment with the brand's authorized post-level analytics. Compare posts by per-post median outcomes and, only if supplied, engagement per impression/reach. Keep platform/account/format/time consistent where possible.

**Not supported:** best posting days, which platform works best, city rankings, follower-normalized engagement, ad ROAS, causal effects, Pakistan-wide market prevalence.

**Scale decision:** Benchmarking above uses repeated real records as a computational load test only, not extra unique observations. For this 2k-row task Pandas is appropriate; evaluate Spark when requirements exceed available single-machine compute/storage or there is a distributed processing need.

**Model:** Held-out TF-IDF macro-F1={f1_score(y_test,pred,average='macro'):.3f}; majority macro-F1={f1_score(y_test,base_pred,average='macro'):.3f}. This is source-domain validation, not verified fashion deployment accuracy.
"""
(OUTPUT/'EXECUTIVE_SUMMARY.md').write_text(review_text,encoding='utf-8')
print(review_text)

## 11 · Export deliverables and teaching rubric
Output contains clean records, manual-review candidates, topic screen, charts, model artifact, holdout errors, benchmark and executive readout. Original dataset CSV remains a **separate standalone file**. Submission rubric: data understanding (1), cleaning (1), statistics (2), plots/interpretation (2), scaling (2), analytics concept/recommendation (2).

In [ ]:
import zipfile
bundle=Path('pakistan_fashion_intelligence_outputs.zip')
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUTPUT.glob('*')):
        if f.is_file():z.write(f,arcname=f.name)
print('Saved',bundle,round(bundle.stat().st_size/1024),'KiB')
print('Output directory:',OUTPUT.resolve())
# Google Colab: from google.colab import files; files.download(str(bundle))